# 1장 — 글자를 숫자로, 확률표로 글쓰기 (실습)

교재 `docs/book/01-char-tokenizer-bigram.md` 와 함께 본다. 이 노트북에서 하는 것:

1. 문자 토크나이저로 코퍼스 114만 자를 정수 열로 바꾸고, 한글 어휘의 특징을 본다
2. "직전 글자 → 다음 글자" 횟수를 세어 (V, V) 확률표를 만들고, 그 표만으로 글을 생성한다
3. 모델이 얼마나 좋은지 숫자 하나(loss)로 재고, 문맥을 늘리면 무엇이 좋아지고 무엇이 터지는지 본다

> 전체 실행 약 2~3분. 가장 오래 걸리는 셀은 §3 의 n-gram 비교(1분 남짓)다.

## 1. 문자 토크나이저

In [ ]:
import math
import time

import torch

from shllm.config import TOKENIZER_DIR, ensure_data_dirs, setup_cpu
from shllm.data import load_corpus
from shllm.tokenizer import CharTokenizer

setup_cpu()
ensure_data_dirs()
text = load_corpus("korean-classics")
tok = CharTokenizer.from_text(text)  # 코퍼스에 나온 글자를 정렬해 0부터 번호를 붙인다
V = tok.vocab_size
print(f"코퍼스 {len(text):,} 자 → 어휘 V = {V:,} 종")

In [ ]:
sample = "옛날 옛적에 호랑이가"
ids = tok.encode(sample)
print(ids)
print(tok.decode(ids))
assert tok.decode(ids) == sample  # round-trip: 잃는 정보가 없어야 한다
print({ch: tok.stoi[ch] for ch in "가나다 ."})  # stoi = string → int

토크나이저는 학습이 끝난 뒤 생성할 때도 **같은 것**을 써야 한다 (번호가 달라지면 모델 출력이 엉뚱한 글자가 된다).
그래서 파일로 저장해 둔다 — 이후 장들은 이 파일을 읽어 쓴다.

In [ ]:
path = TOKENIZER_DIR / "char.json"
tok.save(path)
print(path, "→", CharTokenizer.load(path).vocab_size, "종")

### 1.1 한글 어휘는 무엇으로 이루어졌나

In [ ]:
from collections import Counter

is_hangul = lambda ch: 0xAC00 <= ord(ch) <= 0xD7A3  # 완성형 한글 음절 범위 (가~힣)
is_hanja = lambda ch: 0x4E00 <= ord(ch) <= 0x9FFF  # CJK 통합 한자
hangul = [c for c in tok.chars if is_hangul(c)]
hanja = [c for c in tok.chars if is_hanja(c)]
other = V - len(hangul) - len(hanja)
print(f"한글 음절 {len(hangul):,} / 가능한 11,172  |  한자 {len(hanja):,}  |  기타(공백·기호·영문) {other}")

counts = Counter(text)
total = len(text)
for k in (100, 300, 1000):
    covered = sum(n for _, n in counts.most_common(k))
    print(f"가장 흔한 {k:>4}자가 전체의 {covered / total:6.1%} 를 차지")
print(f"딱 한 번 나온 글자: {sum(1 for n in counts.values() if n == 1):,} 종")

어휘의 4할 가까이가 한자인데, 정작 텍스트의 99% 는 1,000자 안에서 나온다. 즉 **어휘의 대부분은 거의 안 쓰이는 글자**다.
이 "긴 꼬리"가 이 장의 모델을 계속 괴롭힌다 — 한 번도 못 본 글자 조합을 어떻게 다룰 것인가.

## 2. 바이그램 — 확률표 하나로 글쓰기

전체 텍스트를 정수 열로 바꾸고, 앞 90% 를 **학습(train)**, 뒤 10% 를 **검증(val)** 으로 나눈다.
모델은 train 만 보고 만든다. val 은 "처음 보는 글"에서 얼마나 잘 맞히는지 재는 용도다 — 시험 문제를 미리 보여주면 안 되니까.

In [ ]:
data = torch.tensor(tok.encode(text), dtype=torch.int64)  # (N,)
n_train = int(0.9 * len(data))
train, val = data[:n_train], data[n_train:]
print("train", tuple(train.shape), "val", tuple(val.shape), "dtype", data.dtype)

In [ ]:
from shllm.bigram import BigramModel

t0 = time.perf_counter()
bigram = BigramModel(V).fit(train)
print(f"세기: {(time.perf_counter() - t0) * 1000:.0f} ms  (파이썬 루프 없이 index_put_ 한 번)")
print("counts", tuple(bigram.counts.shape), "| 총 쌍 수", int(bigram.counts.sum()), "= len(train) - 1")
print("probs ", tuple(bigram.probs.shape), "| 행 합", bigram.probs.sum(dim=1)[:3])

횟수표를 직접 들여다본다. `'호'` 다음에 무엇이 많이 오나?

In [ ]:
def top_next(model, context: str, k: int = 8):
    p = model.next_token_probs(tok.encode(context))  # (V,)
    v, i = torch.topk(p, k)
    return [(tok.itos[int(j)], round(float(q), 3)) for q, j in zip(v, i)]


for ctx in ("호", "그", "다", " "):
    print(repr(ctx), "→", top_next(bigram, ctx))

가장 흔한 글자 40개끼리의 확률표를 그림으로 본다. 밝을수록 자주 이어지는 쌍이다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 라벨을 찍으려면 CJK 폰트가 필요하다. 시스템에 있는 Noto CJK 를 찾아 쓴다
cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]
matplotlib.rcParams["axes.unicode_minus"] = False

common = [ch for ch, _ in counts.most_common(41) if ch != "\n"][:40]
idx = torch.tensor([tok.stoi[c] for c in common])
sub = bigram.probs[idx][:, idx]  # (40, 40) — 행: 현재 글자, 열: 다음 글자

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(sub.log().clamp(min=-8), cmap="Blues")  # log 로 보면 작은 값 차이도 보인다
labels = ["␣" if c == " " else c for c in common]  # 공백은 보이게 표시
ax.set_xticks(range(40), labels, fontsize=8)
ax.set_yticks(range(40), labels, fontsize=8)
ax.set_xlabel("다음 글자")
ax.set_ylabel("현재 글자")
ax.set_title("P(다음 | 현재), 흔한 40자")
plt.show()

### 2.1 생성 — 확률표에서 주사위 던지기

In [ ]:
g = torch.Generator().manual_seed(1337)  # 시드 고정: 같은 결과를 재현하려고
out = bigram.generate(tok.stoi["옛"], max_new_tokens=300, generator=g)
print(tok.decode(out))

글자 하나 앞만 보니 두세 글자 단위로는 그럴듯한데("하는가", "있는") 문장이 되지 않는다.
그래도 **아무 글자나 찍는 것과는 확실히 다르다**. 그 차이를 숫자로 재 보자.

### 2.2 얼마나 좋은가 — loss

모델이 실제 텍스트의 매 글자에 준 확률을 log 로 바꿔 평균한 뒤 부호를 뒤집은 것이 **loss** (평균 음의 로그가능도)다.
정답 글자에 확률 1 을 주면 0, 확률을 낮게 줄수록 커진다. `exp(loss)` 는 **perplexity** — "매 순간 몇 개 중에 하나를 찍는 셈인가".
아무 정보 없이 V 개 중 균등하게 찍으면 loss = log V 다.

In [ ]:
print(f"균등 찍기  loss = log V = {math.log(V):.3f}   (perplexity {V})")
print(f"bigram     train {bigram.loss(train):.3f}   val {bigram.loss(val):.3f}   (perplexity {bigram.perplexity(val):.1f})")

### 2.3 스무딩 — 한 번도 못 본 쌍

`BigramModel(V, smoothing=α)` 는 모든 칸에 가짜 횟수 α 를 더한다. 0 이면 학습에 없던 쌍이 val 에 나오는 순간 log 0 = -∞ 로 터진다.
교과서는 α=1 (라플라스) 을 쓰라고 하는데, 어휘가 2,800자면 **행마다 가짜 횟수 2,800개**가 생긴다. 실제 횟수가 수십인 행에서는 재앙이다.

In [ ]:
for a in (0.0, 1.0, 0.1, 0.01, 0.001):
    m = BigramModel(V, smoothing=a).fit(train)
    g = torch.Generator().manual_seed(1337)
    sample = tok.decode(m.generate(tok.stoi["옛"], 40, generator=g)).replace("\n", " ")
    print(f"α={a:<6} train {m.loss(train):6.3f}  val {m.loss(val):6.3f}   {sample!r}")

α=1 의 생성문이 한자 범벅인 이유가 여기 있다: 가짜 횟수가 실제를 묻어 버려 "아무 글자나" 에 가까워진다.
α 를 줄이면 val loss 가 내려가다가 어느 지점(0.01 근처)부터 다시 오른다 — 너무 작으면 못 본 쌍에 벌점이 과해진다.
**하이퍼파라미터를 val 로 고르는 것**, 이것이 앞으로 모든 장에서 반복될 패턴이다.

## 3. 문맥을 늘리면? — n-gram

직전 1글자가 아니라 2, 3, 4글자를 보면 당연히 더 잘 맞힐 것이다. 문제는 표의 크기다:
가능한 문맥이 V^(n-1) 가지라 n=3 만 돼도 800만 행, n=4 면 226억 행 — 텐서로는 못 든다.
`NGramModel` 은 **실제로 등장한 문맥만** dict 에 담고, 못 본 긴 문맥은 짧은 문맥의 답으로 물러난다(교재 1.7).

In [ ]:
from shllm.bigram import NGramModel

prompt = tok.encode("옛날 옛적에 호랑이가")
rows = []
for n in (1, 2, 3, 4, 5):
    t0 = time.perf_counter()
    m = NGramModel(n, V).fit(train)
    fit_s = time.perf_counter() - t0
    tr, va = m.loss(train), m.loss(val)
    g = torch.Generator().manual_seed(1337)
    sample = tok.decode(m.generate(prompt, 120, generator=g)).replace("\n", " ")
    rows.append((n, m.n_contexts, V ** (n - 1), tr, va))
    print(f"n={n}  문맥 {m.n_contexts:>9,} / 가능 {V ** (n - 1):.1e}  train {tr:.3f}  val {va:.3f}  ({fit_s:.0f}s)")
    print("    ", sample)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ns = [r[0] for r in rows]
ax.plot(ns, [r[3] for r in rows], "o-", label="train")
ax.plot(ns, [r[4] for r in rows], "s-", label="val")
ax.axhline(math.log(V), ls="--", c="gray", label="균등 (log V)")
ax.set_xlabel("n (문맥 = n-1 글자)")
ax.set_ylabel("loss (nat)")
ax.set_xticks(ns)
ax.legend()
plt.show()

n=5 의 문장이 제일 그럴듯하다. 그런데 정말 "만든" 문장일까? 생성문 안에서 **코퍼스에 토씨 하나 안 틀리고 그대로 있는 가장 긴 조각**을 찾아 본다.
n 이 커질수록 이 조각이 길어진다 — 생성문은 코퍼스 조각을 이어 붙인 것에 가까워진다.

In [ ]:
def longest_copied(sample: str, corpus: str) -> str:
    """sample 의 부분 문자열 중 corpus 에 그대로 존재하는 가장 긴 것. 길이에 대해 이진 탐색한다."""
    lo, hi, best = 0, len(sample), ""
    while lo < hi:
        mid = (lo + hi + 1) // 2
        hit = next((sample[i : i + mid] for i in range(len(sample) - mid + 1) if sample[i : i + mid] in corpus), None)
        if hit is None:
            hi = mid - 1
        else:
            lo, best = mid, hit
    return best


train_text = tok.decode(train.tolist())
for n in (2, 3, 5, 7):
    g = torch.Generator().manual_seed(1337)
    sample = tok.decode(NGramModel(n, V).fit(train).generate(prompt, 120, generator=g))[len("옛날 옛적에 호랑이가") :]
    frag = longest_copied(sample, train_text)
    print(f"n={n}: 베낀 최장 조각 {len(frag):>3}자  {frag!r}")

두 곡선이 갈라지는 지점을 본다.

- **train loss 는 계속 내려간다** — 문맥이 길수록 학습 텍스트를 "외운다". n=5 의 train loss ≈ 1.0 은 학습 텍스트 안에서는 매 글자를 거의 세 개 중 하나로 맞힌다는 뜻이다.
- **val loss 는 어느 n 부터 멈추거나 오른다** — 처음 보는 글에서는 긴 문맥이 학습에 없었던 경우가 대부분이라 결국 짧은 문맥으로 물러난다.
  이것이 **과적합(overfitting)** 이고, 표에 빈 칸이 너무 많은 것(**희소성, sparsity**)이 원인이다.

n=5 에서 등장한 문맥 수와 가능한 문맥 수를 비교하면 표의 몇 %가 채워졌는지 감이 온다.

In [ ]:
n, seen, possible, *_ = rows[-1]
print(f"n={n}: 채워진 문맥 {seen:,} / 가능 {possible:.2e} = {seen / possible:.2e}")
print("→ 표의 사실상 전부가 빈 칸이다. 문맥을 더 늘리는 방법으로는 여기가 끝이다.")

## 정리

- 텍스트 → 정수 열은 `CharTokenizer` 한 줄이면 되지만, 한글은 어휘가 수천 종이라 **긴 꼬리**가 생긴다 (2장에서 BPE 로 어휘 단위를 바꾼다).
- 언어모델 = P(다음 | 지금까지). 가장 단순한 구현은 **세어서 나누기**(바이그램). 생성은 그 확률에서 주사위 던지기, 평가는 loss 하나.
- 문맥을 늘리면 좋아지지만 표는 V^(n-1) 로 폭발하고 대부분 빈 칸이다. **"비슷한 문맥끼리 정보를 공유"** 하려면 문맥을 표의 키가 아니라
  **숫자 벡터** 로 다뤄야 한다 — 그것이 3장(임베딩)과 4장(신경망)이 하는 일이다.

---
**다음 장**: 2장 — BPE 토크나이저. 글자보다 큰 단위로 토큰을 잡아 어휘 크기와 시퀀스 길이를 맞바꾼다.